# Code mẫu — Classification metrics

Mỗi metric phân loại cần một loại đầu vào riêng: nhãn để đếm đúng sai, score để xếp hạng, xác suất để chấm mức tự tin. Các ví dụ dưới đây bắt đầu bằng việc đọc bốn ô của confusion matrix, rồi mới tới chọn ngưỡng, so thứ tự với mức tự tin, và cuối cùng là bài toán nhiều lớp.

## Ví dụ 1: Một hệ thống cảnh báo đang bỏ sót ai?

Có tám buổi học, trong đó ba buổi thật sự cần hỗ trợ (`1`). Các score dưới đây là số cho sẵn để kiểm cách tính metric, chưa phải output của một model nào. Score từ 0.5 trở lên thì hệ thống phát cảnh báo.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

y = np.array([0, 1, 0, 1, 0, 0, 1, 0])
score = np.array([0.1, 0.9, 0.7, 0.6, 0.4, 0.2, 0.3, 0.8])
prediction = score >= 0.5
cm = confusion_matrix(y, prediction, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()          # Hàng là nhãn thật, cột là nhãn dự đoán.
assert (tn, fp, fn, tp) == (3, 2, 1, 2)
print(cm)
print(f"Precision={precision_score(y, prediction):.3f}, recall={recall_score(y, prediction):.3f}, F1={f1_score(y, prediction):.3f}")

rare_y = np.array([1, 1] + [0] * 18)
always_zero = np.zeros(len(rare_y), dtype=int)
print(f"Luôn đoán 0: accuracy={accuracy_score(rare_y, always_zero):.2f}, recall={recall_score(rare_y, always_zero):.2f}")

[[3 2]
 [1 2]]
Precision=0.500, recall=0.667, F1=0.571
Luôn đoán 0: accuracy=0.90, recall=0.00


Đọc ma trận theo hàng: hàng trên là ba buổi không cần hỗ trợ được để yên và hai buổi bị báo nhầm, hàng dưới là một buổi bị bỏ sót và hai buổi được phát hiện. Trong bốn lần cảnh báo có hai lần đúng, nên precision là 0.5. Trong ba buổi cần hỗ trợ, hệ thống bắt được hai, nên recall là `2/3`. Dòng cuối cho thấy vì sao accuracy dễ đánh lừa khi lớp dương hiếm: luôn đoán 0 vẫn đạt accuracy 0.90, nhưng recall bằng 0, tức không phát hiện được buổi nào cần giúp.

## Ví dụ 2: Chọn ngưỡng trên validation, rồi giữ nguyên khi chấm test

Vẫn tám buổi validation như trên. Ta thử ba ngưỡng định sẵn là `0.3, 0.5, 0.7`, và chọn ngưỡng có F1 cao nhất. Nếu hòa thì lấy ngưỡng đứng trước trong danh sách. Sáu buổi test riêng chỉ dùng để chấm ngưỡng đã chốt, không dùng để chọn.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

y_valid = np.array([0, 1, 0, 1, 0, 0, 1, 0])
p_valid = np.array([0.1, 0.9, 0.7, 0.6, 0.4, 0.2, 0.3, 0.8])
thresholds = [0.3, 0.5, 0.7]
scores = []
for threshold in thresholds:
    prediction = p_valid >= threshold    # Score bằng đúng ngưỡng cũng được tính là dương.
    score = f1_score(y_valid, prediction, zero_division=0)
    scores.append(score)
    print(f"t={threshold:.1f}: cảnh báo={int(prediction.sum())}, valid F1={score:.3f}")
chosen = thresholds[int(np.argmax(scores))]
print("Ngưỡng chốt bằng validation:", chosen)

y_test = np.array([1, 0, 1, 0, 0, 1])
p_test = np.array([0.8, 0.2, 0.4, 0.35, 0.1, 0.25])
test_prediction = p_test >= chosen
print(f"Test F1={f1_score(y_test, test_prediction):.3f}")
print(confusion_matrix(y_test, test_prediction, labels=[0, 1]))

t=0.3: cảnh báo=6, valid F1=0.667
t=0.5: cảnh báo=4, valid F1=0.571
t=0.7: cảnh báo=3, valid F1=0.333
Ngưỡng chốt bằng validation: 0.3
Test F1=0.667
[[2 1]
 [1 2]]


Hạ ngưỡng xuống 0.3 thì bắt đủ cả ba buổi cần hỗ trợ của validation, đổi lại số cảnh báo tăng lên sáu. Trên test, ngưỡng ấy vẫn bỏ sót một buổi. Đó là lỗi trên những mẫu khác, và ta chấp nhận con số ấy như nó là. Nếu thấy test chưa đẹp mà quay lại đổi ngưỡng, thì tập test đã bị dùng để chọn, và không còn độc lập nữa. Ngưỡng cũng là một phần của model, nên phải được chọn trên validation như mọi tham số khác.

## Ví dụ 3: Xếp đúng thứ tự khác với tự tin đúng mức

Bốn mẫu gồm hai nhãn 1 và hai nhãn 0, trong đó hai score bằng nhau ở 0.8. Ta tính ROC-AUC bằng tay, theo mọi cặp gồm một mẫu dương và một mẫu âm, với cặp hòa được nửa điểm. Sau đó bình phương các score: thứ tự giữ nguyên, nhưng mức xác suất thay đổi.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, log_loss

y = np.array([1, 0, 1, 0])
p = np.array([0.8, 0.8, 0.4, 0.2])
positive = p[y == 1]
negative = p[y == 0]
points = (positive[:, None] > negative[None, :]).astype(float)
points += 0.5 * (positive[:, None] == negative[None, :])
print("Điểm bốn cặp:", points.ravel().tolist())
print(f"AUC tính tay={points.mean():.3f}, thư viện={roc_auc_score(y, p):.3f}")
for name, probability in [("gốc", p), ("bình phương", p ** 2)]:
    print(f"{name}: AUC={roc_auc_score(y, probability):.3f}, log loss={log_loss(y, probability):.3f}")

Điểm bốn cặp: [0.5, 1.0, 0.0, 1.0]
AUC tính tay=0.625, thư viện=0.625
gốc: AUC=0.625, log loss=0.743
bình phương: AUC=0.625, log loss=0.835


Bốn cặp được 0.5, 1, 0 và 1 điểm, nên AUC là `2.5 / 4 = 0.625`, khớp với thư viện. Bình phương các số trong khoảng 0 đến 1 không làm đổi thứ tự hay các cặp hòa, nên AUC giữ nguyên. Log loss thì tăng từ 0.743 lên 0.835, vì mức xác suất đã thay đổi. AUC chỉ đo model xếp hạng có đúng không, chứ không chứng nhận rằng score có thể đọc như một xác suất.

## Ví dụ 4: Đọc lỗi của một classifier ba lớp

Iris là bộ dữ liệu có sẵn trong scikit-learn. Mỗi hàng là một bông hoa, với bốn feature là chiều dài và chiều rộng của đài hoa và cánh hoa, tính bằng cm. Target là một trong ba loài. Ta dùng một Pipeline gồm scaler và LogisticRegression, giữ lại 30 bông làm validation, rồi đọc F1 của từng loài.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score

iris = load_iris()
X_train, X_valid, y_train, y_valid = train_test_split(
    iris.data, iris.target, test_size=30, stratify=iris.target, random_state=51,
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300))
model.fit(X_train, y_train)
prediction = model.predict(X_valid)
classifier = model.named_steps["logisticregression"]
probability = model.predict_proba(X_valid)
assert np.allclose(probability.sum(axis=1), 1)
assert classifier.classes_.tolist() == [0, 1, 2]
print("Thứ tự lớp:", iris.target_names[classifier.classes_].tolist())
print(confusion_matrix(y_valid, prediction, labels=classifier.classes_))
print("F1 từng lớp:", np.round(f1_score(y_valid, prediction, average=None), 3).tolist())
print(f"Macro-F1={f1_score(y_valid, prediction, average='macro'):.3f}")

Thứ tự lớp: ['setosa', 'versicolor', 'virginica']
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
F1 từng lớp: [1.0, 0.947, 0.952]
Macro-F1=0.967


Mỗi hàng của confusion matrix ứng với một loài thật, theo thứ tự đã in ở dòng đầu. Chỉ có một bông versicolor bị đoán thành virginica, và lỗi ấy làm giảm F1 của cả hai loài: versicolor mất một mẫu đúng, còn virginica nhận thêm một mẫu sai. Setosa không bị nhầm lần nào. Macro-F1 0.967 là trung bình đều của ba con số F1. Đó là kết quả của một lần chia 30 bông, chưa đủ để nói model sẽ luôn tốt như vậy.